# 08 · Master Pipeline Kaggle — Zero-Trust / Zero-Persistent-Image

| Componente | Localização | Volátil? |
|---|---|---|
| INPUT (paste/upload/img2img) | `/dev/shm/comfy_ui_input` | ✅ RAM |
| OUTPUT (imagens geradas) | `/dev/shm/comfy_ui_output` | ✅ RAM |
| TEMP (latentes, previews) | `/dev/shm/comfy_ui_temp` | ✅ RAM |
| USER (config, workflows) | `/dev/shm/comfy_ui_user` | ✅ RAM |
| LOGS (comfyui.log) | `/dev/shm/comfy_ui_logs` | ✅ RAM |
| ZIP AES-256 de download | `/dev/shm/comfy_ui_archive/output.zip` | ✅ RAM |

**Nenhuma imagem toca `/kaggle/working` em nenhum momento controlado pelo pipeline.**

O ZIP de download existe **apenas em `/dev/shm`**. Não há staging em `/kaggle/working`.
O download é feito diretamente do path `/dev/shm/comfy_ui_archive/output.zip` via
o painel Files do Kaggle Notebook ou via `IPython.display.FileLink`.

**SECURE_MODE=True por padrão:**
- Manager PERMITIDO (state/downloads/custom nodes em /dev/shm)
- ngrok PERMITIDO (após health check, token via Kaggle Secrets)
- reuse_existing=False (sempre processo novo)
- A segurança vem do isolamento de filesystem, não do bloqueio de funcionalidade

**Limites explícitos:**
- custom nodes executam Python arbitrário — não são uma barreira de segurança
- ngrok cria exposição externa
- custom nodes podem fazer requests externos e acessar dados em memória
- não controla infraestrutura Kaggle, acesso privilegiado do provedor,
  nem vulnerabilidades em dependências externas.

## Fluxo obrigatório por sessão
1. Validar Dataset montado em `/kaggle/input/<slug>`
2. Sincronizar repo GitHub
3. Detectar GPU
4. Drive setup (credenciais apagadas após montagem)
5. `record_working_snapshot()` + `assert_working_clean()` — verificação inicial
6. Provisionar tmpfs + instalar ComfyUI (Manager ON, isolado em /dev/shm)
7. Snapshot de custom nodes + `rebuild_working_snapshot_after_provisioning()` (baseline = ambiente provisionado)
8. `assert_invariants()` — verificação de paths (fail-closed)
9. Iniciar ComfyUI (processo novo, reuse=False, Manager ON, ngrok ON)
10. `assert_comfy_process_isolation()` — verificar isolamento do processo
11. **GENERATE → COLLECT → ZIP AES-256 em /dev/shm → DOWNLOAD → DELETE → VERIFY**
12. `audit_working_directory()` (auditoria consolidada — cobre `assert_working_policy()` + `assert_only_allowed_persistent_artifact()`)

In [ ]:
from pathlib import Path
import subprocess, sys, os, time, json

REPO_URL = "https://github.com/automadevs/colab-pipeline.git"
WORKDIR  = Path("/kaggle/working")
REPO_DIR = WORKDIR / "colab-pipeline"
SCRIPTS_DIR = WORKDIR / "scripts"
COMFYUI_DIR = WORKDIR / "ComfyUI"

# Todos os dirs de I/O em tmpfs (RAM volátil)
SHM_INPUT   = Path("/dev/shm/comfy_ui_input")
SHM_OUTPUT  = Path("/dev/shm/comfy_ui_output")
SHM_TEMP    = Path("/dev/shm/comfy_ui_temp")
SHM_ARCHIVE = Path("/dev/shm/comfy_ui_archive")
SHM_USER    = Path("/dev/shm/comfy_ui_user")
SHM_LOGS    = Path("/dev/shm/comfy_ui_logs")
# ZIP existe APENAS em /dev/shm — nunca em /kaggle/working
SECURE_ZIP  = SHM_ARCHIVE / "output.zip"

# Modelos (binários de modelo, não imagens)
MODELS_DIR  = COMFYUI_DIR / "models"

# SECURE_MODE: Manager e ngrok PERMITIDOS com isolamento de filesystem.
SECURE_MODE = True
os.environ["COMFYUI_SECURE_MODE"] = "1" if SECURE_MODE else "0"

def _get_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

def resolve_dataset_name(override=None):
    if override:
        return override
    username = _get_secret("KAGGLE_USERNAME")
    dataset_name = _get_secret("KAGGLE_DATASET_NAME")
    if not username or not dataset_name:
        raise ValueError(
            "Dataset Kaggle não resolvido. Configure Secrets:\n"
            "  - KAGGLE_USERNAME\n  - KAGGLE_DATASET_NAME"
        )
    return f"{username}/{dataset_name}"

DATASET_OVERRIDE = None
DATASET = resolve_dataset_name(DATASET_OVERRIDE)
DRIVE_BASE = "Automa/ComfyUI"

print(f"[INFO] Dataset: {DATASET}")
print(f"[INFO] SECURE_MODE={os.environ.get('COMFYUI_SECURE_MODE')}")

In [ ]:
# Validar Dataset montado como Input
DATASET_SLUG = _get_secret("KAGGLE_DATASET_NAME")
if not DATASET_SLUG:
    raise RuntimeError("Secret KAGGLE_DATASET_NAME não configurado.")

DATASET_INPUT_DIR = Path("/kaggle/input") / DATASET_SLUG
if not DATASET_INPUT_DIR.is_dir():
    raise RuntimeError(
        f"Dataset '{DATASET}' não está anexado como Input deste notebook "
        f"(esperado em {DATASET_INPUT_DIR}). "
        "Add Input → Datasets → busque o dataset → Add."
    )

print(f"[INFO] Dataset montado (somente leitura): {DATASET_INPUT_DIR}")

In [ ]:
# Sincronizar repositório e copiar scripts
import importlib, shutil

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if SCRIPTS_DIR.exists():
    shutil.rmtree(SCRIPTS_DIR)
shutil.copytree(REPO_DIR / "scripts", SCRIPTS_DIR)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

importlib.invalidate_caches()
for mod in ("comfyui_setup", "ngrok_tunnel", "gpu_detect", "kaggle_drive_sync"):
    m = sys.modules.get(mod)
    if m is not None:
        importlib.reload(m)

print("[INFO] Scripts:", sorted(p.name for p in SCRIPTS_DIR.glob("*.py")))

In [ ]:
# GPU detection
from gpu_detect import detect_gpu
GPU_INFO = detect_gpu()
print(json.dumps(GPU_INFO, indent=2, ensure_ascii=False))
if not GPU_INFO.get("has_gpu"):
    raise RuntimeError("GPU NVIDIA não detectada. Ative GPU no Kaggle.")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run(["df", "-h", str(WORKDIR)], check=False)
subprocess.run(["df", "-h", "/dev/shm"], check=False)

In [ ]:
# Google Drive — SOMENTE persistência/backup. Credenciais apagadas após montagem.
from kaggle_drive_sync import test_drive_connection, cleanup_rclone_credentials

DRIVE_AVAILABLE = False
try:
    test_res = test_drive_connection(drive_base=DRIVE_BASE, env="kaggle")
    if test_res["status"] == "pass":
        DRIVE_AVAILABLE = True
        DRIVE_PATH = Path(test_res["drive_path"])
        # Apagar credenciais imediatamente após montagem
        cleanup_rclone_credentials()
        print(f"[INFO] Drive montado: {DRIVE_PATH}")
    else:
        print(f"[WARN] Drive não disponível: {test_res.get('error')}")
except Exception as e:
    print(f"[WARN] Drive: {e}")

In [ ]:
# ============================================================
# PRÉ-VERIFICAÇÃO DE SEGURANÇA
# Garante que /kaggle/working está limpo ANTES de iniciar
# ============================================================
from comfyui_setup import (
    assert_no_persistent_images, SecurityError, final_filesystem_check,
    record_working_snapshot, assert_working_clean,
)

print("[SECURITY] Verificação inicial de /kaggle/working...")
assert_no_persistent_images(label="PRE-CHECK")
record_working_snapshot()
assert_working_clean()
print("[SECURITY] /kaggle/working limpo — pode continuar")

In [ ]:
# Provisionar tmpfs + instalar ComfyUI (Manager ON, isolado em /dev/shm)
from comfyui_setup import (
    setup_comfyui, provision_shm_dirs,
    check_custom_nodes_allowlist, snapshot_custom_nodes,
    rebuild_working_snapshot_after_provisioning,
)

CUSTOM_NODES = [
    "cubiq/ComfyUI_essentials",
    "lbouaraba/comfyui-krea2edit",
]

provision_shm_dirs(SHM_INPUT, SHM_OUTPUT, SHM_TEMP, SHM_ARCHIVE, SHM_USER, SHM_LOGS)
subprocess.run(["df", "-h", "/dev/shm"], check=False)

setup_comfyui(
    comfyui_dir=COMFYUI_DIR,
    models_dir=MODELS_DIR,
    custom_nodes=CUSTOM_NODES,
    output_dir=SHM_OUTPUT,
    input_dir=SHM_INPUT,
    temp_dir=SHM_TEMP,
    additional_model_roots=[("dataset_models", DATASET_INPUT_DIR)],
    strict_allowlist=True,
    # Manager PERMITIDO em SECURE_MODE (state/downloads em /dev/shm)
    enable_manager=True,
)

# Snapshot dos custom nodes APÓS setup — usado para detectar alterações
CUSTOM_NODES_SNAPSHOT = snapshot_custom_nodes(COMFYUI_DIR)
print(f"[SECURITY] Custom nodes snapshot: {list(CUSTOM_NODES_SNAPSHOT['nodes'].keys())}")
# Re-baseline do working snapshot: o baseline passa a ser o ambiente
# TOTALMENTE PROVISIONADO (custom nodes + extra_model_paths.yaml). A fase
# de geracao nao pode persistir nada novo a partir daqui. Fail-closed:
# aborta sem reescrever o baseline se houver artefato sensivel.
rebuild_working_snapshot_after_provisioning(label="POST-SETUP")
print(f"[SECURITY] INPUT  → {SHM_INPUT}")
print(f"[SECURITY] OUTPUT → {SHM_OUTPUT}")
print(f"[SECURITY] TEMP   → {SHM_TEMP}")
print(f"[SECURITY] ARCHIVE→ {SHM_ARCHIVE}")

In [ ]:
# ============================================================
# VERIFICAÇÃO DE SEGURANÇA (fail-closed)
# INPUT/OUTPUT/TEMP/USER/LOGS/ARCHIVE devem estar em /dev/shm
# ============================================================
from comfyui_setup import assert_shm_path, assert_invariants, SecurityError

print("=== COMFYUI SECURITY CONFIGURATION ===")
for label, path in [("INPUT", SHM_INPUT), ("OUTPUT", SHM_OUTPUT),
                    ("TEMP", SHM_TEMP), ("USER", SHM_USER),
                    ("LOGS", SHM_LOGS), ("ARCHIVE", SHM_ARCHIVE)]:
    assert_shm_path(path, label)
    print(f"{label:8s} = {path}")
print("=== END SECURITY CONFIGURATION ===")

# Validar invariantes antes do start
assert_invariants(
    input_dir=SHM_INPUT, output_dir=SHM_OUTPUT, temp_dir=SHM_TEMP,
    user_dir=SHM_USER, log_dir=SHM_LOGS, archive_dir=SHM_ARCHIVE,
)
print(f"[SECURITY] assert_invariants: PASS — pipeline pode continuar")

In [ ]:
# Iniciar ComfyUI (processo NOVO, SECURE_MODE, Manager ON, ngrok ON)
from comfyui_setup import (
    start_comfyui_runtime, provision_shm_dirs,
    assert_comfy_process_isolation,
)

COMFYUI_PORT = 8188
# ngrok: PERMITIDO em SECURE_MODE (token via Kaggle Secrets)
ENABLE_NGROK = True

provision_shm_dirs(SHM_INPUT, SHM_OUTPUT, SHM_TEMP, SHM_ARCHIVE, SHM_USER, SHM_LOGS)

try:
    runtime = start_comfyui_runtime(
        comfyui_dir=COMFYUI_DIR,
        host="127.0.0.1",
        port=COMFYUI_PORT,
        output_dir=SHM_OUTPUT,
        input_dir=SHM_INPUT,
        temp_dir=SHM_TEMP,
        user_dir=SHM_USER,
        enable_ngrok=ENABLE_NGROK,
        enable_manager=True,  # Manager ON (state em /dev/shm)
        reuse_existing=False,
    )
except Exception as e:
    from comfyui_setup import secure_cleanup
    secure_cleanup(raise_on_persistent=False)
    raise

if not runtime["health"]:
    from comfyui_setup import secure_cleanup
    secure_cleanup(raise_on_persistent=False)
    raise RuntimeError(f"ComfyUI não ficou saudável. Log: {runtime['log_path']}")

# Verificar isolamento do processo
process_info = assert_comfy_process_isolation(
    pid=runtime.get("pid"), port=COMFYUI_PORT,
    input_dir=SHM_INPUT, output_dir=SHM_OUTPUT,
    temp_dir=SHM_TEMP, user_dir=SHM_USER, log_dir=SHM_LOGS,
)
print(f"PID: {runtime.get('pid')} | GPU: {GPU_INFO.get('gpu_count', 0)} GPU(s)")
print(f"Manager: ON | ngrok: {'ON' if runtime.get('ngrok_started') else 'OFF'}")
if runtime.get('public_url'):
    print(f"Public URL: {runtime['public_url']}")

In [ ]:
# ============================================================
# GERENCIAMENTO DE CICLO DE VIDA
# Fluxo obrigatório: GENERATE → COLLECT → ZIP AES-256 → (DOWNLOAD) → DELETE → VERIFY
# Execute APÓS geração. Use try/finally para garantir cleanup mesmo em erro.
# ============================================================
import gc, urllib.request
from comfyui_setup import (
    create_secure_zip, cleanup_zip, clear_input, clear_output,
    assert_no_persistent_images, final_filesystem_check,
    secure_cleanup, SecurityError, verify_custom_nodes_unchanged,
    assert_working_policy, assert_only_allowed_persistent_artifact,
    audit_working_directory, secure_persistent_write,
)

COMFYUI_API = f"http://127.0.0.1:{COMFYUI_PORT}"

def _api_get(path):
    with urllib.request.urlopen(f"{COMFYUI_API}{path}", timeout=10) as r:
        return json.loads(r.read())

def wait_queue_empty(poll_s=3.0, timeout_s=7200.0):
    start = time.time()
    while True:
        q = _api_get("/queue")
        running = len(q.get("queue_running", []))
        pending = len(q.get("queue_pending", []))
        elapsed = int(time.time() - start)
        print(f"[POLL] running={running} pending={pending} elapsed={elapsed}s  ", end="\r")
        if running == 0 and pending == 0:
            print(f"\n[INFO] Fila drenada após {elapsed}s.")
            return
        if elapsed > timeout_s:
            raise TimeoutError(f"Fila não drenou em {timeout_s}s")
        time.sleep(poll_s)

def report_history_failures():
    history = _api_get("/history")
    failures = [pid for pid, e in history.items()
                if e.get("status", {}).get("status_str") == "error"]
    if failures:
        print(f"[WARN] {len(failures)} job(s) com falha: {failures}")
    else:
        print(f"[INFO] Histórico OK: {len(history)} job(s)")

secure_zip_path = None
try:
    # 1. Verificar custom nodes antes de coletar
    changes = verify_custom_nodes_unchanged(COMFYUI_DIR, CUSTOM_NODES_SNAPSHOT, strict=True)

    # 2. Esperar fila drenar
    wait_queue_empty()
    report_history_failures()

    # 3. Verificar que /kaggle/working não tem artefatos ANTES do ZIP
    # Auditoria consolidada: coleta TODAS as categorias antes de reportar
    # (imagens + policy + snapshot-diff) num unico relatorio — evita ver
    # uma categoria por run.
    audit_working_directory(label="PRE-ZIP")

    # 4. Criar ZIP AES-256 em /dev/shm (senha do Secret SECRET_ZIP_PASSWORD)
    secure_zip_path = create_secure_zip(
        src_dir=SHM_OUTPUT,
        archive_dir=SHM_ARCHIVE,
        zip_name="output.zip",
        run_encryption_test=True,
    )
    print(f"[SECURITY] ZIP pronto em: {secure_zip_path}")
    print(f"[SECURITY] Para baixar: abra o painel Files do Kaggle e navegue até {SHM_ARCHIVE}")

    # 5. Exportar cópia final persistente via secure_persistent_write
    #    (única escrita permitida em /kaggle/working)
    if secure_zip_path and secure_zip_path.exists():
        secure_persistent_write(secure_zip_path)
        print(f"[SECURITY] output_secure.zip copiado para /kaggle/working")

    # 6. Limpar output e input APÓS criar ZIP
    clear_output(SHM_OUTPUT)
    clear_input(SHM_INPUT)
    gc.collect()

    # 7. Verificação final: /kaggle/working sem imagens + apenas output_secure.zip
    # Auditoria consolidada (relatorio unico com todas as categorias).
    audit_working_directory(label="POST-CLEAR")
    print("[SECURITY] Ciclo de vida concluído — imagens apenas em ZIP (tmpfs)")

except (SecurityError, Exception) as _lifecycle_err:
    print(f"[ERROR] Falha no ciclo de vida: {_lifecycle_err}")
    secure_cleanup(raise_on_persistent=False)
    raise

In [ ]:
# ============================================================
# DOWNLOAD DO ZIP
# O ZIP está em /dev/shm/comfy_ui_archive/output.zip (RAM volátil)
# output_secure.zip também foi copiado para /kaggle/working via secure_persistent_write
#
# Para baixar, use uma das opções:
#   1. Painel Files do Kaggle → navegue até /dev/shm/comfy_ui_archive/
#   2. IPython FileLink (renderiza link clicável no notebook)
#   3. Baixe /kaggle/working/output_secure.zip (cópia persistente)
# ============================================================
from IPython.display import FileLink, display

if secure_zip_path and secure_zip_path.exists():
    size_mb = secure_zip_path.stat().st_size / (1024**2)
    print(f"[INFO] ZIP disponível: {secure_zip_path} ({size_mb:.1f} MB)")
    print("[INFO] Clique no link abaixo para baixar (tmpfs):")
    display(FileLink(str(secure_zip_path)))
    print()
    persistent_zip = WORKDIR / "output_secure.zip"
    if persistent_zip.exists():
        psize_mb = persistent_zip.stat().st_size / (1024**2)
        print(f"[INFO] output_secure.zip em /kaggle/working ({psize_mb:.1f} MB)")
        display(FileLink(str(persistent_zip)))
    print()
    print("[SECURITY] Após o download, execute a célula de cleanup abaixo.")
else:
    print("[WARN] ZIP não encontrado — execute a célula de ciclo de vida primeiro.")


In [ ]:
# ============================================================
# CLEANUP PÓS-DOWNLOAD — execute APÓS confirmar download
# ============================================================
from comfyui_setup import (
    cleanup_zip, final_filesystem_check, safe_remove,
    assert_no_persistent_images, SecurityError,
)

try:
    # Apagar ZIP de tmpfs
    if secure_zip_path:
        cleanup_zip(secure_zip_path)

    # Verificação final de filesystem
    result = final_filesystem_check(scan_root=WORKDIR, silent=False)
    if result["violations"] > 0:
        raise SecurityError(
            f"SECURITY VIOLATION: {result['violations']} artefato(s) sensível(is) "
            "encontrado(s) em /kaggle/working após cleanup."
        )

    print("[SECURITY] ✅ Cleanup pós-download: zero artefatos persistentes")

except SecurityError:
    raise
except Exception as e:
    print(f"[ERROR] Cleanup falhou: {e}")
    raise

In [ ]:
# Validação de workflow (CLIPLoader krea2)
WORKFLOW_JSON = WORKDIR / "lustify_simple_t2i.json"

def validate_workflow_node(path, node_id="9", expected_type="krea2"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Workflow não encontrado: {path}")
    wf = json.loads(path.read_text(encoding="utf-8"))
    node = None
    if node_id in wf and isinstance(wf[node_id], dict):
        node = wf[node_id]
    elif isinstance(wf.get("nodes"), list):
        node = next((n for n in wf["nodes"] if str(n.get("id")) == node_id), None)
    if node is None:
        raise KeyError(f"Nó {node_id} ausente")
    actual = node.get("class_type") or node.get("type")
    if actual != expected_type:
        raise ValueError(f"Nó {node_id}: type={actual!r}, esperado {expected_type!r}")
    print(f"[OK] Nó {node_id}: type={actual!r}")
    return True

validate_workflow_node(WORKFLOW_JSON)

In [ ]:
%%bash
# Auditoria operacional de filesystem
echo "=== TMPFS (/dev/shm) ==="
df -h /dev/shm
echo
echo "=== FOOTPRINT VOLÁTIL ==="
du -sh /dev/shm/comfy_ui_{input,output,temp,archive} 2>/dev/null || echo "(dirs não criados)"
echo
echo "=== ZERO-DISK AUDIT: imagens em /kaggle/working ==="
COUNT=$(find /kaggle/working -type f \( -iname '*.png' -o -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.webp' -o -iname '*.gif' -o -iname '*.bmp' -o -iname '*.tif' -o -iname '*.tiff' -o -iname '*.latent' \) 2>/dev/null | wc -l)
echo "Image files em /kaggle/working: $COUNT"
if [ "$COUNT" -gt 0 ]; then
    echo "SECURITY FAIL: imagens encontradas:"
    find /kaggle/working -type f \( -iname '*.png' -o -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.webp' \) 2>/dev/null | head -20
else
    echo "STATUS: PASS (zero imagens)"
fi
echo
echo "=== ZERO-DISK AUDIT: ZIPs em /kaggle/working ==="
find /kaggle/working -type f -iname '*.zip' 2>/dev/null | head -10
echo
echo "=== ComfyUI subpaths críticos ==="
for d in /kaggle/working/ComfyUI/input /kaggle/working/ComfyUI/output /kaggle/working/ComfyUI/temp; do
    if [ -d "$d" ]; then
        count=$(find "$d" -type f 2>/dev/null | wc -l)
        echo "$d: $count arquivo(s)"
    else
        echo "$d: não existe"
    fi
done
echo
echo "=== SYMLINKS em /kaggle/working ==="
find /kaggle/working -maxdepth 4 -type l 2>/dev/null | head -10 || echo "(nenhum)"

In [ ]:
# Push de logs ao Drive (SOMENTE logs — sem imagens, sem ZIP)
# Logs estão em /dev/shm/comfy_ui_logs — sync de logs é seguro (não são imagens)
from kaggle_drive_sync import sync_outputs

log_file = SHM_LOGS / "comfyui.log"
has_logs = log_file.exists() and log_file.stat().st_size > 0

if has_logs and DRIVE_AVAILABLE:
    try:
        sync_res = sync_outputs(
            action="push",
            categories=["logs"],
            local_logs=SHM_LOGS,
            drive_base=DRIVE_BASE,
            env="kaggle",
        )
        print(f"[INFO] Logs: {sync_res['synced']} enviado(s), {sync_res['skipped']} inalterado(s)")
    except Exception as e:
        print(f"[WARN] Push logs: {e}")
else:
    print("[INFO] Drive não disponível ou sem logs. Pronto para gerar!")

In [ ]:
# ============================================================
# VERIFICAÇÃO FINAL DE FILESYSTEM
# Execute ao encerrar a sessão
# ============================================================
from comfyui_setup import (
    final_filesystem_check, SecurityError,
    assert_working_policy, assert_only_allowed_persistent_artifact,
    audit_working_directory, secure_cleanup,
)

# Auditoria consolidada: um unico relatorio com todas as categorias.
# (assert_working_policy/assert_only_allowed_persistent_artifact seguem
# disponiveis e sao cobertos por esta chamada. A categoria
# final_filesystem fica DESLIGADA por padrao: output_secure.zip eh o
# artefato permitido pos-geracao.)
audit_working_directory(label="FINAL")

# Verificações finais (audit apos cleanup)
secure_cleanup(raise_on_persistent=False)
# Auditoria consolidada: aqui a categoria final_filesystem FICA LIGADA,
# porque no ENCERRAMENTO da sessao nem o output_secure.zip deve restar
# (ja cobre assert_working_policy() e assert_only_allowed_persistent_artifact(),
# importados acima para compatibilidade/documentacao dos guardrails).
audit_working_directory(label="FINAL", include_final_filesystem=True)
result = final_filesystem_check(scan_root=WORKDIR, silent=False)

if result["violations"] > 0:
    raise SecurityError(
        f"SECURITY VIOLATION: {result['violations']} artefato(s) em /kaggle/working.\n"
        "Apague os arquivos listados acima antes de encerrar."
    )
print("[SECURITY] Verificação final: PASS")

## Próximo passo

ComfyUI rodando com **Manager ON, ngrok ON, zero-persistent-image**.
Nenhum artefato de geração toca `/kaggle/working` exceto `output_secure.zip`.

**Riscos residuais documentados:**
- custom nodes executam Python arbitrário e podem fazer requests externos
- ngrok cria exposição externa (túnel público)
- não há egress control no ambiente Kaggle
- custom nodes podem acessar dados em memória

Fluxo por sessão:
1. Gere pela interface (outputs → `/dev/shm/comfy_ui_output`)
2. Execute **ciclo de vida** (espera fila → ZIP AES-256 em `/dev/shm` → `output_secure.zip` em `/kaggle/working`)
3. Execute **download** (FileLink de `/dev/shm` ou baixe `output_secure.zip`)
4. Execute **cleanup pós-download** (apaga ZIP + verifica filesystem)
5. Execute **verificação final** (confirma zero artefatos proibidos em `/kaggle/working`)

Para logs com Google Drive: `09_sync_outputs.ipynb`

## [Opcional] Civitai → Local: adicionar modelo

Baixa binários de modelo (`.safetensors`, `.ckpt`) diretamente para `MODELS_DIR/<categoria>/`.
**NÃO baixa imagens.** Execute manualmente, não no Run All.

Requer: Célula 4 (sync repo) executada + Secret `CIVITAI_TOKEN`.

In [ ]:
os.environ.setdefault("KAGGLE_USERNAME", _get_secret("KAGGLE_USERNAME") or "")
os.environ.setdefault("KAGGLE_DATASET_NAME", _get_secret("KAGGLE_DATASET_NAME") or "")

from kaggle_dataset_manager import (
    CATEGORIES, collect_input_queue, download_resolved_queue,
    queue_contains_checkpoint, resolve_queue_metadata,
)

for _cat in CATEGORIES:
    (MODELS_DIR / _cat).mkdir(parents=True, exist_ok=True)

CIVITAI_TOKEN = _get_secret("CIVITAI_TOKEN") or _get_secret("CIVITAI_API_KEY")
if not CIVITAI_TOKEN:
    raise RuntimeError("CIVITAI_TOKEN não encontrado.")

print("=" * 60)
print(f"CIVITAI → LOCAL | Destino: {MODELS_DIR}")
print("=" * 60)

_pending = collect_input_queue()
if not _pending:
    print("[INFO] Nenhum item informado.")
else:
    _resolved = resolve_queue_metadata(_pending, CIVITAI_TOKEN)
    if _resolved:
        _ckpt_dest = None
        if queue_contains_checkpoint(_resolved):
            _c = input("Checkpoint: 1=checkpoints/ 2=diffusion_models/: ").strip()
            _ckpt_dest = {"1": "checkpoints", "2": "diffusion_models"}.get(_c, _c)
        _downloaded = download_resolved_queue(
            _resolved, staging_dir=MODELS_DIR,
            token=CIVITAI_TOKEN, checkpoint_destination=_ckpt_dest,
        )
        print(f"\n[SUCCESS] {len(_downloaded)} arquivo(s) em {MODELS_DIR}")